# 🧬 Pandemic Early Warning · Unified Pipeline
## One accession in → Full dual-signal anomaly report out

### DNA-level (DNABERT-2) + Protein-level (ESM2 6-frame) + Fused scoring + BLAST triage

---

| Stage | Description |
|---|---|
| **0** | Install & Config — only cell you need to edit |
| **1** | Download from ENA |
| **2** | Parse & Quality Filter |
| **3** | K-mer Baseline (Isolation Forest) |
| **4** | DNABERT-2 DNA Embeddings |
| **5** | 6-Frame Translation (fasta_translate logic) |
| **6** | ESM2 Protein Embeddings (monitor logic) |
| **7** | Protein Isolation Forest |
| **8** | Dual-Signal Fusion Score |
| **9** | UMAP Visualisation (DNA + Protein + Fused) |
| **10** | BLAST Triage (BLASTn + BLASTp) |
| **11** | Summary Report & Export |

> Press `Runtime → Run All`. Everything is automatic.


## Stage 0 · Install

In [1]:
import subprocess, sys

def run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '⚠️ '} {label or cmd[:60]}")
    return r

run("apt-get install -qq -y ncbi-blast+", "BLAST+")

pkgs = [
    "umap-learn", "biopython",
    "transformers==4.40.0", "accelerate", "huggingface_hub",
    "scikit-learn", "numpy", "pandas", "matplotlib", "tqdm",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ All packages installed")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception:
    DRIVE_AVAILABLE = False
    print("ℹ️  Local mode — outputs saved to /content/outputs/")


✅ BLAST+
✅ All packages installed
Mounted at /content/drive


## Stage 0 · ⚙️ CONFIG
> **This is the only cell you need to edit.**
> Change the accession number, press `Runtime → Run All`, wait for results.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
#  ✏️  EDIT THESE — everything else is automatic
# ═══════════════════════════════════════════════════════════════════════════

SRA_ACCESSION    = "SRR37006656"   # ← your SRA accession
MAX_READS        = 10_000          # ← reads to analyse (10K=fast, 100K=production)
EMBED_SUBSAMPLE  = 2_000           # ← reads for DNABERT-2 (raise with more GPU time)
BLAST_TOP_N      = 5               # ← anomalies to BLAST per mode (BLASTn + BLASTp)

# ═══════════════════════════════════════════════════════════════════════════
#  🔧 ADVANCED — safe to leave as-is
# ═══════════════════════════════════════════════════════════════════════════
SEED              = 42
DNABERT_MODEL     = "zhihan1996/DNABERT-2-117M"
ESM2_MODEL        = "facebook/esm2_t30_150M_UR50D"
EMBED_SUBSAMPLE  = 100    # was 2_000
EMBED_BATCH_DNA  = 8      # was 256
EMBED_BATCH_PROT = 8      # was 64            # ESM2 batch
UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.1
UMAP_METRIC       = "cosine"
IF_CONTAMINATION  = 0.1
NOVEL_THRESHOLD   = 90.0           # % id below which = novel strain
MIN_PROTEIN_LEN   = 30             # min amino acids to keep from 6-frame translation

import os, numpy as np, random, torch
DRIVE = '/content/drive/MyDrive/pandemic_outputs' if DRIVE_AVAILABLE else '/content/outputs'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs('/content/chunks', exist_ok=True)
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
FASTQ_PATH = f"{SRA_ACCESSION}.fastq"

print(f"✅ Config ready")
print(f"   Accession : {SRA_ACCESSION}  |  Reads: {MAX_READS:,}  |  Device: {DEVICE.upper()}")
print(f"   Output    : {DRIVE}")


✅ Config ready
   Accession : SRR37006656  |  Reads: 10,000  |  Device: CUDA
   Output    : /content/drive/MyDrive/pandemic_outputs


## Stage 1 · Download from ENA

In [3]:
import requests, subprocess, os

print(f"🔍 Resolving {SRA_ACCESSION} via ENA API …")
r = requests.get("https://www.ebi.ac.uk/ena/portal/api/filereport",
    params={"accession": SRA_ACCESSION, "result": "read_run",
            "fields": "fastq_ftp", "format": "json"})
r.raise_for_status()
ftp_paths = r.json()[0]["fastq_ftp"].split(";")
print(f"   Found {len(ftp_paths)} file(s): {[p.split('/')[-1] for p in ftp_paths]}")

gz_path = f"{SRA_ACCESSION}_R1.fastq.gz"
url     = "https://" + ftp_paths[0]

if not os.path.exists(gz_path):
    print(f"⬇️  Downloading {url.split('/')[-1]} …")
    subprocess.run(f"wget -q --show-progress -O '{gz_path}' '{url}'",
                   shell=True, check=True)
else:
    print(f"✅ Already downloaded ({os.path.getsize(gz_path)/1e6:.1f} MB)")

if not os.path.exists(FASTQ_PATH) or os.path.getsize(FASTQ_PATH) < 1000:
    print(f"📦 Extracting first {MAX_READS:,} reads …")
    subprocess.run(f"zcat '{gz_path}' | head -n {MAX_READS*4} > '{FASTQ_PATH}'",
                   shell=True, check=True)

with open(FASTQ_PATH) as f:
    n_total = sum(1 for line in f) // 4
print(f"✅ {FASTQ_PATH}  →  {n_total:,} reads")


🔍 Resolving SRR37006656 via ENA API …
   Found 2 file(s): ['SRR37006656_1.fastq.gz', 'SRR37006656_2.fastq.gz']
⬇️  Downloading SRR37006656_1.fastq.gz …
📦 Extracting first 10,000 reads …
✅ SRR37006656.fastq  →  10,000 reads


## Stage 2 · Parse, Quality Filter & Split

In [4]:
import numpy as np

def parse_fastq(filepath, max_reads=None):
    reads = []
    with open(filepath) as f:
        for i, line in enumerate(f):
            if i % 4 == 1:
                reads.append(line.strip())
                if max_reads and len(reads) >= max_reads:
                    break
    return reads

def quality_filter(reads, min_length=50, max_n_fraction=0.10):
    clean = [r for r in reads
             if len(r) >= min_length
             and r.count('N') / len(r) <= max_n_fraction]
    print(f"   Quality filter: {len(reads):,} → {len(clean):,} "
          f"(removed {len(reads)-len(clean):,})")
    return clean

all_reads = quality_filter(parse_fastq(FASTQ_PATH, max_reads=MAX_READS))

gc = np.array([(s.count('G')+s.count('C'))/len(s) for s in all_reads])
classified_reads   = [r for r,g in zip(all_reads,gc) if 0.35<=g<=0.65]
unclassified_reads = [r for r,g in zip(all_reads,gc) if g<0.35 or g>0.65]
unclassified_rate  = len(unclassified_reads) / len(all_reads)

print(f"\n📊 Split summary  [GC-proxy — swap for Kraken2 if DB available]")
print(f"   Classified   : {len(classified_reads):,}")
print(f"   Unclassified : {len(unclassified_reads):,}  ({unclassified_rate*100:.1f}%)")


   Quality filter: 10,000 → 9,612 (removed 388)

📊 Split summary  [GC-proxy — swap for Kraken2 if DB available]
   Classified   : 9,380
   Unclassified : 232  (2.4%)


## Stage 3 · K-mer Baseline

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import IsolationForest
import numpy as np

vec = CountVectorizer(analyzer='char', ngram_range=(4,4), max_features=500)
X_cls_kmer   = vec.fit_transform(classified_reads).toarray().astype(float)
X_uncls_kmer = vec.transform(unclassified_reads).toarray().astype(float)

iso_kmer = IsolationForest(contamination=IF_CONTAMINATION, random_state=SEED, n_jobs=-1)
iso_kmer.fit(X_cls_kmer)

scores_kmer_cls   = iso_kmer.score_samples(X_cls_kmer)
scores_kmer_uncls = iso_kmer.score_samples(X_uncls_kmer)
threshold_kmer    = np.percentile(scores_kmer_cls, 10)
tail_count_kmer   = (scores_kmer_uncls < threshold_kmer).sum()
tail_pct_kmer     = tail_count_kmer / len(scores_kmer_uncls) * 100
delta_kmer        = scores_kmer_uncls.mean() - scores_kmer_cls.mean()

print(f"✅ K-mer baseline")
print(f"   Separation Δ    : {delta_kmer:.4f}")
print(f"   Anomalous tail  : {tail_count_kmer:,}  ({tail_pct_kmer:.1f}%)")


✅ K-mer baseline
   Separation Δ    : -0.0124
   Anomalous tail  : 121  (52.2%)


## Stage 4 · Load DNABERT-2
> Flash attention disabled automatically.

In [8]:
import os

model_dir = "/root/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0"

print("Files in snapshot:")
for f in sorted(os.listdir(model_dir)):
    print(f"  {f}")

Files in snapshot:


FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0'

In [7]:
import torch, sys, numpy as np
from transformers import AutoTokenizer, AutoConfig
from tqdm.auto import tqdm

MODEL_DIR = "/root/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0"

tokenizer_dna = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

# ── Find the already-loaded bert_layers module in sys.modules ──────────────
bert_layers_mod = None
for name, mod in sys.modules.items():
    if 'bert_layers' in name and 'zhihan' in name:
        bert_layers_mod = mod
        print(f"Found: {name}")
        break

if bert_layers_mod is None:
    # Force it to load by triggering AutoConfig
    AutoConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
    for name, mod in sys.modules.items():
        if 'bert_layers' in name and 'zhihan' in name:
            bert_layers_mod = mod
            print(f"Found after config load: {name}")
            break

# ── List available classes ─────────────────────────────────────────────────
print("\nClasses in bert_layers:")
for k in dir(bert_layers_mod):
    obj = getattr(bert_layers_mod, k)
    if isinstance(obj, type):
        print(f"  {k}")

OSError: Incorrect path_or_model_id: '/root/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch, sys, numpy as np
from transformers import AutoTokenizer, AutoConfig
from tqdm.auto import tqdm

MODEL_DIR = "/root/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0"

tokenizer_dna = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

# ── Get BertModel directly from cached module ──────────────────────────────
bert_layers_mod = None
for name, mod in sys.modules.items():
    if 'bert_layers' in name and 'zhihan' in name:
        bert_layers_mod = mod
        break

BertModel = bert_layers_mod.BertModel

# ── Load config + null flash attn ─────────────────────────────────────────
config = AutoConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
config.__dict__.update({
    'use_flash_attn': False,
    '_attn_implementation': 'eager',
    'attn_implementation': 'eager',
    'pad_token_id': 0,
})

# Null flash attn in module before model instantiation
for name, mod in sys.modules.items():
    if 'bert_layers' in name and 'zhihan' in name:
        mod.flash_attn_qkvpacked_func = None
        print(f"✅ Nulled flash_attn")

# ── Instantiate + load weights ─────────────────────────────────────────────
model_dna = BertModel(config)
state_dict = torch.load(os.path.join(MODEL_DIR, "pytorch_model.bin"),
                        map_location="cpu")
# Strip 'bert.' prefix if present
from collections import OrderedDict
clean = OrderedDict(
    (k[5:] if k.startswith('bert.') else k, v)
    for k, v in state_dict.items()
)
missing, unexpected = model_dna.load_state_dict(clean, strict=False)
print(f"✅ Weights loaded  |  missing={len(missing)}  unexpected={len(unexpected)}")

model_dna.eval().to(DEVICE)

def embed_dna(sequences, batch_size=EMBED_BATCH_DNA, max_length=512):
    all_vecs = []
    for i in tqdm(range(0, len(sequences), batch_size),
                  desc="DNA embed", unit="batch", leave=False):
        batch = sequences[i:i+batch_size]
        enc   = tokenizer_dna(batch, return_tensors='pt', padding=True,
                              truncation=True, max_length=max_length).to(DEVICE)
        with torch.no_grad():
            out = model_dna(**enc)
        hidden = out.last_hidden_state if hasattr(out, 'last_hidden_state') else out[0]
        mask   = enc['attention_mask'].unsqueeze(-1).float()
        vecs   = (hidden * mask).sum(1) / mask.sum(1)
        all_vecs.append(vecs.cpu().numpy())
    return np.vstack(all_vecs)

print(f"✅ DNABERT-2 ready on {DEVICE.upper()}")

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"DEVICE value   : {DEVICE}")
print(f"Model device   : {next(model_dna.parameters()).device}")

# Force to GPU if available
if torch.cuda.is_available():
    model_dna = model_dna.to('cuda')
    DEVICE = 'cuda'
    print(f"✅ Moved to GPU: {next(model_dna.parameters()).device}")
else:
    print("⚠️  No GPU detected — check Runtime → Change runtime type → T4 GPU")

## Stage 4 · DNABERT-2 Embeddings + Isolation Forest

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

rng       = np.random.default_rng(SEED)
uncls_idx = rng.choice(len(unclassified_reads),
                        size=min(EMBED_SUBSAMPLE, len(unclassified_reads)), replace=False)
cls_idx   = rng.choice(len(classified_reads),
                        size=min(500, len(classified_reads)), replace=False)
uncls_sample = [unclassified_reads[i] for i in uncls_idx]
cls_sample   = [classified_reads[i]   for i in cls_idx]

print(f"Embedding {len(uncls_sample):,} unclassified reads …")
X_uncls_emb = embed_dna(uncls_sample)
print(f"Embedding {len(cls_sample):,} classified reads …")
X_cls_emb   = embed_dna(cls_sample)

iso_dna = IsolationForest(contamination=IF_CONTAMINATION, random_state=SEED, n_jobs=-1)
iso_dna.fit(X_cls_emb)
scores_dna_cls   = iso_dna.score_samples(X_cls_emb)
scores_dna_uncls = iso_dna.score_samples(X_uncls_emb)

threshold_dna  = np.percentile(scores_dna_cls, 10)
tail_count_dna = (scores_dna_uncls < threshold_dna).sum()
tail_pct_dna   = tail_count_dna / len(scores_dna_uncls) * 100
delta_dna      = scores_dna_uncls.mean() - scores_dna_cls.mean()

print(f"\n✅ DNABERT-2 Isolation Forest")
print(f"   Separation Δ    : {delta_dna:.4f}  (k-mer: {delta_kmer:.4f})")
print(f"   Anomalous tail  : {tail_count_dna:,}  ({tail_pct_dna:.1f}%)")
print(f"   Beats k-mer?    : {'✅ YES' if delta_dna < delta_kmer else '❌ NO (honest negative)'}")


## Stage 5 · 6-Frame Translation
Translates all unclassified reads in 6 reading frames (fasta_translate.py logic).
Deduplicates and filters proteins > 30 amino acids.

In [ ]:
from Bio.Seq import Seq
import numpy as np
from tqdm.auto import tqdm

def translate_6_frames(dna_str, min_len=MIN_PROTEIN_LEN):
    """Translate a DNA string in all 6 frames. Returns list of protein strings."""
    seq = Seq(dna_str.replace('N', 'A'))   # replace N with A for translation
    proteins = []
    # Forward frames
    for i in range(3):
        prot = str(seq[i:].translate(to_stop=True))
        if len(prot) >= min_len:
            proteins.append(prot)
    # Reverse complement frames
    rc = seq.reverse_complement()
    for i in range(3):
        prot = str(rc[i:].translate(to_stop=True))
        if len(prot) >= min_len:
            proteins.append(prot)
    return proteins

print(f"Translating {len(unclassified_reads):,} unclassified reads in 6 frames …")
print(f"   Min protein length: {MIN_PROTEIN_LEN} aa")

seen_proteins  = set()
all_proteins   = []          # unique protein strings
protein_source = []          # which read index each protein came from

for read_idx, read_seq in enumerate(tqdm(unclassified_reads, desc="Translating")):
    frames = translate_6_frames(read_seq)
    for prot in frames:
        if prot not in seen_proteins:
            seen_proteins.add(prot)
            all_proteins.append(prot)
            protein_source.append(read_idx)

print(f"\n✅ 6-frame translation complete")
print(f"   Input reads      : {len(unclassified_reads):,}")
print(f"   Unique proteins  : {len(all_proteins):,}")
print(f"   Avg per read     : {len(all_proteins)/len(unclassified_reads):.1f}")


## Stage 6 · Load ESM2 Protein Model

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, EsmModel
from tqdm.auto import tqdm

print(f"Loading {ESM2_MODEL} …")
tokenizer_prot = AutoTokenizer.from_pretrained(ESM2_MODEL)
model_prot     = EsmModel.from_pretrained(ESM2_MODEL).to(DEVICE)

# fp16 on GPU for speed, fp32 on CPU for stability
if DEVICE == "cuda":
    model_prot = model_prot.half()
model_prot.eval()

def embed_proteins(sequences, batch_size=EMBED_BATCH_PROT,
                   max_length=512, chunk_dir='/content/chunks'):
    """
    Embed protein sequences with ESM2.
    Saves chunks to disk every 500 batches (monitor.py RAM-protection logic).
    Returns full embedding matrix.
    """
    import os
    all_vecs    = []
    chunk_count = 0
    chunk_files = []

    for i in tqdm(range(0, len(sequences), batch_size),
                  desc="Protein embed", unit="batch", leave=False):
        batch = sequences[i:i+batch_size]
        enc   = tokenizer_prot(batch, return_tensors='pt', padding=True,
                               truncation=True, max_length=max_length).to(DEVICE)
        with torch.no_grad():
            out = model_prot(**enc)
        vecs = out.last_hidden_state.mean(dim=1).cpu().float().numpy()
        all_vecs.append(vecs)

        # Flush to disk every 500 batches (RAM protection)
        if len(all_vecs) >= 500:
            chunk_path = os.path.join(chunk_dir, f"emb_chunk_{chunk_count:03d}.npy")
            np.save(chunk_path, np.vstack(all_vecs))
            chunk_files.append(chunk_path)
            all_vecs    = []
            chunk_count += 1

    # Final chunk
    if all_vecs:
        chunk_path = os.path.join(chunk_dir, f"emb_chunk_final.npy")
        np.save(chunk_path, np.vstack(all_vecs))
        chunk_files.append(chunk_path)

    # Reload all chunks into memory
    full_matrix = np.vstack([np.load(c) for c in chunk_files])
    return full_matrix

print(f"✅ ESM2 ready on {DEVICE.upper()} "
      f"({'fp16' if DEVICE=='cuda' else 'fp32'})")


## Stage 6 · ESM2 Embeddings + Protein Isolation Forest

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

print(f"Embedding {len(all_proteins):,} unique proteins with ESM2 …")
X_prot_emb = embed_proteins(all_proteins)
print(f"✅ Protein embedding matrix: {X_prot_emb.shape}")

# ── Isolation Forest in protein space ─────────────────────────────────────
# Reference = bottom 80% by L2 norm (proxy for background proteome)
norms   = np.linalg.norm(X_prot_emb, axis=1)
ref_idx = np.argsort(norms)[:int(len(norms)*0.8)]

iso_prot = IsolationForest(contamination=0.05, random_state=SEED, n_jobs=-1)
iso_prot.fit(X_prot_emb[ref_idx])
scores_prot = iso_prot.score_samples(X_prot_emb)

threshold_prot  = np.percentile(scores_prot, 5)
tail_count_prot = (scores_prot < threshold_prot).sum()
tail_pct_prot   = tail_count_prot / len(scores_prot) * 100

print(f"\n✅ ESM2 Isolation Forest")
print(f"   Proteins scored    : {len(scores_prot):,}")
print(f"   Anomalous tail (5%): {tail_count_prot:,}  ({tail_pct_prot:.1f}%)")


## Stage 7 · Dual-Signal Fusion
Combines DNA-level (DNABERT-2) and protein-level (ESM2) anomaly scores.
Maps protein anomaly scores back to source reads, then fuses 50/50.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ── Normalise both to [0,1]  (lower = more anomalous) ─────────────────────
scaler   = MinMaxScaler()
dna_norm = scaler.fit_transform(scores_dna_uncls.reshape(-1,1)).flatten()

scaler2   = MinMaxScaler()
prot_norm = scaler2.fit_transform(scores_prot.reshape(-1,1)).flatten()

# ── Map protein scores → source reads (min score per read) ────────────────
# protein_source[i] = which read produced protein i
read_prot_score = np.ones(len(uncls_sample))  # default: most normal (1.0)

for prot_idx, read_idx in enumerate(protein_source):
    # uncls_sample uses uncls_idx mapping back to unclassified_reads
    # find which position in uncls_sample this read_idx corresponds to
    if prot_idx >= len(prot_norm):
        break
    # Find sample position
    original_read = unclassified_reads[read_idx]
    try:
        sample_pos = uncls_sample.index(original_read)
        if prot_norm[prot_idx] < read_prot_score[sample_pos]:
            read_prot_score[sample_pos] = prot_norm[prot_idx]
    except ValueError:
        pass   # read not in subsample — skip

# ── Fused score ────────────────────────────────────────────────────────────
fused_scores  = 0.5 * dna_norm + 0.5 * read_prot_score
top_fused_idx = np.argsort(fused_scores)[:50]

print("✅ Fusion complete")
print(f"   DNA anomalous (10%)     : {(dna_norm < 0.1).sum():,} reads")
print(f"   Protein anomalous (5%)  : {tail_count_prot:,} ORFs")
print(f"   Fused top-50 candidates : {len(top_fused_idx)}")
print(f"   Fusion weight           : 50% DNABERT-2 + 50% ESM2")


## Stage 8 · Tri-Panel UMAP

In [ ]:
import umap, matplotlib.pyplot as plt, numpy as np

print("UMAP: DNA space …")
red_dna  = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                      metric=UMAP_METRIC, random_state=SEED, n_jobs=1, verbose=False)
emb_dna  = red_dna.fit_transform(X_uncls_emb)

n_prot_plot = min(2000, len(X_prot_emb))
rng         = np.random.default_rng(SEED)
prot_plot_idx = rng.choice(len(X_prot_emb), size=n_prot_plot, replace=False)
print(f"UMAP: protein space ({n_prot_plot:,} ORFs) …")
red_prot = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                      metric=UMAP_METRIC, random_state=SEED, n_jobs=1, verbose=False)
emb_prot = red_prot.fit_transform(X_prot_emb[prot_plot_idx])

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.patch.set_facecolor('#0d1117')
for ax in axes: ax.set_facecolor('#0d1117')

def dark_scatter(ax, xy, scores, cmap, title, n=None):
    sc = ax.scatter(xy[:,0], xy[:,1], c=scores, cmap=cmap,
                    s=8, alpha=0.75, edgecolors='none', rasterized=True)
    cb = fig.colorbar(sc, ax=ax, pad=0.01)
    cb.set_label('Anomaly score', color='white', fontsize=8)
    cb.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel("UMAP 1", color='#aaa', fontsize=8)
    ax.set_ylabel("UMAP 2", color='#aaa', fontsize=8)
    ax.tick_params(colors='#555')
    for spine in ax.spines.values(): spine.set_visible(False)

dark_scatter(axes[0], emb_dna,  dna_norm,
             'viridis', f"DNABERT-2 · {len(emb_dna):,} reads")
dark_scatter(axes[1], emb_prot, prot_norm[prot_plot_idx],
             'plasma',  f"ESM2 · {n_prot_plot:,} ORFs")
dark_scatter(axes[2], emb_dna,  fused_scores,
             'inferno', f"Fused Score (50/50)")

# Circle top-50 fused anomalies on panel 3
axes[2].scatter(emb_dna[top_fused_idx,0], emb_dna[top_fused_idx,1],
                s=45, facecolors='none', edgecolors='#e63946',
                linewidths=0.9, label='Top-50 fused', rasterized=True)
axes[2].legend(fontsize=8, framealpha=0.2, labelcolor='white', facecolor='#1a1a2e')

fig.text(0.01, 0.01,
         f"n_neighbors={UMAP_N_NEIGHBORS} · min_dist={UMAP_MIN_DIST} · "
         f"metric={UMAP_METRIC!r} · seed={SEED}",
         color='#444', fontsize=7)
plt.suptitle(f"Dual-Signal UMAP — {SRA_ACCESSION}",
             color='white', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(pad=2)
out = f'{DRIVE}/umap_{SRA_ACCESSION}.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ Saved → {out}")


## Stage 9 · BLAST Triage
BLASTn on top fused reads + BLASTp on top anomalous proteins.

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import SeqIO
import numpy as np, time

# ── Write FASTAs ──────────────────────────────────────────────────────────
BLASTN_FASTA = f"/content/top_reads_{SRA_ACCESSION}.fasta"
BLASTP_FASTA = f"/content/top_proteins_{SRA_ACCESSION}.fasta"

top_reads = [(uncls_sample[i], fused_scores[i]) for i in top_fused_idx
             if uncls_sample[i].count('N')/len(uncls_sample[i]) < 0.05][:BLAST_TOP_N]
with open(BLASTN_FASTA, 'w') as f:
    for rank, (seq, score) in enumerate(top_reads):
        f.write(f">read_rank{rank+1}_score{score:.4f}\n{seq}\n")

top_prot_idx = np.argsort(prot_norm)[:BLAST_TOP_N]
top_prots    = [(all_proteins[i], prot_norm[i]) for i in top_prot_idx
                if i < len(all_proteins)
                and all_proteins[i].count('X')/len(all_proteins[i]) < 0.1]
with open(BLASTP_FASTA, 'w') as f:
    for rank, (seq, score) in enumerate(top_prots):
        f.write(f">protein_rank{rank+1}_score{score:.4f}\n{seq}\n")

print(f"✅ BLASTn FASTA: {len(top_reads)} reads")
print(f"✅ BLASTp FASTA: {len(top_prots)} proteins")

def run_blast(fasta_path, program, db, label):
    records = list(SeqIO.parse(fasta_path, "fasta"))
    results = []
    print(f"\n🔬 {label} ({len(records)} sequences, ~2-3 min each) …")
    for rec in records:
        unit = "bp" if program == "blastn" else "aa"
        print(f"  {rec.id}  ({len(rec.seq)} {unit}) …", end=" ", flush=True)
        try:
            handle = NCBIWWW.qblast(program, db, str(rec.seq),
                                     hitlist_size=1, expect=0.001)
            result = NCBIXML.read(handle)
            if result.alignments:
                aln = result.alignments[0]
                hsp = aln.hsps[0]
                pct = hsp.identities / hsp.align_length * 100
                results.append({"query": rec.id, "hit": aln.title,
                                 "pct_id": pct, "evalue": hsp.expect,
                                 "mode": label})
                print(f"✅  {pct:.1f}% id  e={hsp.expect:.0e}  "
                      f"{aln.title.split('|')[-1].strip()[:50]}")
            else:
                results.append({"query": rec.id, "hit": "No hit",
                                 "pct_id": 0, "evalue": 1, "mode": label})
                print("❌  No hit")
        except Exception as e:
            results.append({"query": rec.id, "hit": str(e),
                             "pct_id": 0, "evalue": 1, "mode": label})
            print(f"⚠️   {e}")
        time.sleep(3)
    return results

blastn_results = run_blast(BLASTN_FASTA, "blastn", "nt",  "BLASTn")
blastp_results = run_blast(BLASTP_FASTA, "blastp", "nr",  "BLASTp")
all_blast      = blastn_results + blastp_results


## Stage 9 · Automated Triage

In [ ]:
HIGH_RISK_FAMILIES = ['Orthomyxoviridae','Coronaviridae','Paramyxoviridae',
                      'Flaviviridae','Filoviridae','Rhabdoviridae']
PATHOGENIC_DOMAINS = ['PB2','Spike','RBD','HA','Hemagglutinin','Fusion protein',
                      'NS3','NS5','Polymerase','Capsid','Envelope','Protease']
CLINICAL_ORGANISMS = ['Naegleria','Acanthamoeba','Cryptosporidium',
                       'Giardia','Toxoplasma']

def triage(hit, pct_id):
    if any(d in hit for d in PATHOGENIC_DOMAINS):  return "🚨 HIGH — Conserved pathogenic domain"
    if any(f in hit for f in HIGH_RISK_FAMILIES):   return "🚨 HIGH — Known pathogenic viral family"
    if "virus" in hit.lower() and pct_id < NOVEL_THRESHOLD:
        return f"⚠️  ELEVATED — Novel viral variant ({pct_id:.1f}% id)"
    if any(c in hit for c in CLINICAL_ORGANISMS):   return "🔍 MEDIUM — Clinically relevant eukaryote"
    if pct_id < NOVEL_THRESHOLD:                     return f"🔍 MEDIUM — Divergent sequence ({pct_id:.1f}% id)"
    return "⬜ ROUTINE — Characterised organism"

print("═"*70)
print(f"  DUAL-SIGNAL TRIAGE REPORT — {SRA_ACCESSION}")
print("═"*70)
for section, results in [("BLASTn  (nucleotide reads)", blastn_results),
                          ("BLASTp  (translated proteins)", blastp_results)]:
    print(f"\n  ── {section} {'─'*(50-len(section))}")
    for r in results:
        label = triage(r['hit'], r['pct_id'])
        name  = r['query'].split('_score')[0]
        hit   = r['hit'].split('|')[-1].strip()[:60]
        print(f"  {name:<28}  {r['pct_id']:5.1f}% id  {label}")
        print(f"    → {hit}")

high  = [r for r in all_blast if "HIGH"     in triage(r['hit'],r['pct_id'])]
elev  = [r for r in all_blast if "ELEVATED" in triage(r['hit'],r['pct_id'])]
novel = [r for r in all_blast if 0 < r['pct_id'] < NOVEL_THRESHOLD]
verdict = "🚨 ALERT" if high else ("⚠️  ELEVATED" if elev else "🔍 MONITOR")

print(f"\n{'═'*70}")
print(f"  High-risk hits    : {len(high)}")
print(f"  Novel/elevated    : {len(elev)}")
print(f"  Divergent (<{NOVEL_THRESHOLD}%) : {len(novel)}")
print(f"  Verdict           : {verdict}")


## Stage 10 · Summary Report

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, pandas as pd

fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.38)

# ── 1. K-mer score distributions ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0,0])
ax1.hist(scores_kmer_cls,   bins=50, alpha=0.65, color='#3182bd', label='Classified')
ax1.hist(scores_kmer_uncls, bins=50, alpha=0.65, color='#e6550d', label='Unclassified')
ax1.axvline(threshold_kmer, color='#222', linestyle='--', linewidth=1.2)
ax1.set_title("K-mer Scores", fontsize=11)
ax1.set_xlabel("IF Score")
ax1.legend(fontsize=8)
ax1.spines[['top','right']].set_visible(False)

# ── 2. DNABERT-2 score distributions ──────────────────────────────────────
ax2 = fig.add_subplot(gs[0,1])
ax2.hist(scores_dna_cls,   bins=50, alpha=0.65, color='#3182bd', label='Classified')
ax2.hist(scores_dna_uncls, bins=50, alpha=0.65, color='#9b59b6', label='Unclassified')
ax2.axvline(threshold_dna, color='#222', linestyle='--', linewidth=1.2)
ax2.set_title("DNABERT-2 Scores", fontsize=11)
ax2.set_xlabel("IF Score")
ax2.legend(fontsize=8)
ax2.spines[['top','right']].set_visible(False)

# ── 3. ESM2 protein score distribution ────────────────────────────────────
ax3 = fig.add_subplot(gs[0,2])
ax3.hist(scores_prot, bins=50, alpha=0.75, color='#e67e22')
ax3.axvline(threshold_prot, color='#222', linestyle='--', linewidth=1.2,
            label=f'5th pct ({threshold_prot:.3f})')
ax3.set_title("ESM2 Protein Scores", fontsize=11)
ax3.set_xlabel("IF Score")
ax3.legend(fontsize=8)
ax3.spines[['top','right']].set_visible(False)

# ── 4. Delta comparison ────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1,0])
lbls  = ["K-mer", "DNABERT-2"]
delts = [delta_kmer, delta_dna]
cols  = ['#3182bd', '#9b59b6']
bars  = ax4.bar(lbls, delts, color=cols, width=0.45, edgecolor='white')
ax4.axhline(0, color='#aaa', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, delts):
    ax4.text(bar.get_x()+bar.get_width()/2, val-0.0006,
             f"{val:.4f}", ha='center', va='top', fontsize=11, fontweight='bold')
ax4.set_ylabel("Separation Delta")
ax4.set_title("DNA Method Comparison", fontsize=11)
ax4.spines[['top','right']].set_visible(False)

# ── 5. Signal complementarity ──────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1,1])
dna_set   = set(np.argsort(dna_norm)[:50])
fused_set = set(top_fused_idx[:50])
dna_only  = len(dna_set - fused_set)
both      = len(dna_set & fused_set)
prot_only = 50 - both - dna_only
ax5.bar(
    ["DNA only", "Both signals", "Protein only"],
    [dna_only, both, prot_only],
    color=['#3182bd', '#9b59b6', '#e67e22'],
    edgecolor='white', linewidth=0.8
)
ax5.set_ylabel("Reads in top-50")
ax5.set_title("Signal Complementarity", fontsize=11)
ax5.spines[['top','right']].set_visible(False)

# ── 6. BLAST results table ─────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1,2])
ax6.axis('off')
if all_blast:
    rows = [
        [r['mode'],
         f"{r['pct_id']:.1f}%",
         f"{r['evalue']:.0e}",
         triage(r['hit'], r['pct_id']).split('—')[0].strip(),
         r['hit'].split('|')[-1].strip()[:30]]
        for r in all_blast
    ]
    tbl = ax6.table(cellText=rows,
                    colLabels=["Mode", "%id", "E-val", "Triage", "Hit"],
                    loc='center', cellLoc='left')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.5)
    tbl.scale(1, 1.5)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#ddd')
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif row <= len(rows) and any(x in rows[row-1][3] for x in ['🚨', '⚠️']):
            cell.set_facecolor('#fff3cd')
        else:
            cell.set_facecolor('#f8f9fa')
ax6.set_title("BLAST Results", fontsize=11, pad=20)

plt.suptitle(
    f"Unified Pipeline Summary — {SRA_ACCESSION}  |  "
    f"Unclassified: {unclassified_rate*100:.1f}%  |  "
    f"K-mer delta={delta_kmer:.4f}  |  DNABERT-2 delta={delta_dna:.4f}  |  "
    f"ESM2 tail={tail_pct_prot:.1f}%  |  Verdict: {verdict}",
    fontsize=10, fontweight='bold', y=1.01
)

out = f'{DRIVE}/summary_{SRA_ACCESSION}.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → {out}")

## Stage 10 · Export Results

In [ ]:
import pandas as pd, numpy as np

# BLAST CSV
pd.DataFrame(all_blast).to_csv(f'{DRIVE}/blast_{SRA_ACCESSION}.csv', index=False)

# Top fused anomalies CSV
pd.DataFrame({
    "rank":        range(1, len(top_fused_idx)+1),
    "read_seq":    [uncls_sample[i][:80] for i in top_fused_idx],
    "dna_score":   dna_norm[top_fused_idx].round(4),
    "fused_score": fused_scores[top_fused_idx].round(4),
}).to_csv(f'{DRIVE}/top_fused_{SRA_ACCESSION}.csv', index=False)

# Summary
summary = pd.DataFrame({
    "Metric": ["Accession","Total reads","Unclassified rate",
                "K-mer Δ","DNABERT-2 Δ","DNABERT-2 beats k-mer",
                "Unique proteins (6-frame)","ESM2 anomalous tail",
                "Novel BLASTn hits","Novel BLASTp hits",
                "High-risk hits","Verdict"],
    "Value":  [SRA_ACCESSION, f"{len(all_reads):,}",
               f"{unclassified_rate*100:.1f}%",
               f"{delta_kmer:.4f}", f"{delta_dna:.4f}",
               "YES" if delta_dna < delta_kmer else "NO (honest negative)",
               f"{len(all_proteins):,}",
               f"{tail_pct_prot:.1f}%  ({tail_count_prot:,} ORFs)",
               str(len([r for r in blastn_results if 0<r['pct_id']<NOVEL_THRESHOLD])),
               str(len([r for r in blastp_results if 0<r['pct_id']<NOVEL_THRESHOLD])),
               str(len(high)), verdict]
})
summary.to_csv(f'{DRIVE}/summary_{SRA_ACCESSION}.csv', index=False)

try:
    from IPython.display import display
    display(summary.style.hide(axis='index'))
except Exception:
    print(summary.to_string(index=False))

print(f"\n🏁 Pipeline complete. Outputs in {DRIVE}/")
print(f"   summary_{SRA_ACCESSION}.csv")
print(f"   blast_{SRA_ACCESSION}.csv")
print(f"   top_fused_{SRA_ACCESSION}.csv")
print(f"   umap_{SRA_ACCESSION}.png")
print(f"   summary_{SRA_ACCESSION}.png")


The numbers
Data:

10,000 reads downloaded, 9,612 passed quality filter (removed 388 N-heavy reads — consistent with the original notebook)
GC-proxy split: 9,380 classified / 232 unclassified (2.4%) — much lower than the original 42.4% from real Kraken2. Expected — GC-proxy is a weak separator

K-mer baseline:

Separation Δ = −0.0124 — slightly stronger than previous runs (−0.0105, −0.0123). Consistent range.
Anomalous tail = 121 reads (52.2%) — high because with only 232 unclassified reads the distribution is noisy

DNABERT-2 (100 reads, CPU):

Separation Δ = −0.0515 — significantly stronger than k-mer
Anomalous tail = 75 reads (75.0%)
Beats k-mer baseline: ✅ YES

This is the most important result in the notebook. With only 100 reads and CPU inference, DNABERT-2 is showing 4× stronger separation than k-mer (−0.0515 vs −0.0124). This is the opposite of the honest negative from earlier runs — and worth examining why.
The likely explanation: 100 reads is too small a sample for stable IF scoring. The classified reference set (500 reads embedded) is 5× larger than the unclassified set (100 reads), which inflates the separation delta artificially. The previous honest negative at 2,000 reads is the more reliable result. Do not present the −0.0515 figure as definitive — note it as a small-sample artifact.
6-frame translation:

232 unclassified reads → 665 unique proteins (2.9 per read average) — clean, expected

ESM2 (665 proteins, CPU fp32):

Embedding matrix: (665, 640) — ESM2-t30 produces 640-dim vectors (not 480 as assumed in bolt-on — worth fixing)
Anomalous tail: 34 ORFs (5.1%) — exactly at the 5% contamination threshold, correct

Fusion:

DNA anomalous (10%): only 1 read — very sparse at 100-read subsample
Protein anomalous: 34 ORFs
Top-50 fused candidates generated


BLAST results — the real biology
BLASTn top hits:
Rank% idOrganismTriage198.0%Brevundimonas phoenicis⬜ ROUTINE273.3%Azospirillum sp.🔍 MEDIUM376.7%Gadus macrocephalus (Pacific cod)🔍 MEDIUM492.0%Gallid alphaherpesvirus 1 (Marek's disease virus)⚠️ FLAG592.7%Gallid alphaherpesvirus 1 (same isolate)⚠️ FLAG
BLASTp: all 5 returned no hit — proteins too short (50 aa) for nr database matching at these read lengths. Expected on 151bp reads.

Key findings to highlight
1. Gallid alphaherpesvirus 1 at ranks 4 and 5 is the standout result. Marek's disease virus (MDV) is a poultry herpesvirus — not directly pathogenic to humans, but its presence in a hospital wastewater sample at 92% identity is genuinely anomalous. Two independent reads hitting the same viral isolate (CSW-1_CUD) strengthens the signal. This warrants follow-up.
2. Gadus macrocephalus (Pacific cod) at rank 3 (76.7% id) is biologically implausible in a New York hospital wastewater sample — 76.7% is low enough that this is likely a divergent environmental sequence with some similarity to fish genomic DNA, not actual cod. Flag as database artefact.
3. Azospirillum sp. at rank 2 (73.3% id) — plant-growth-promoting rhizobacterium. Environmental background, divergent strain, not clinically significant but genuinely novel at that identity.
4. BLASTp returning zero hits is not a failure — it tells you the anomalous ORFs are either too short or sufficiently divergent from the nr database that they have no known protein homologues. That's actually the most interesting region of sequence space for novel pathogen discovery.